# 05 - Belauri forward simulations on the real prepared region

**Real geography; synthetic emissions and meteorology.** The municipality polygon, the road / built-up / facility geometry and the position of the external patch are real (prepared in `data/belauri`). Every emission rate, the external onset and every wind is a **synthetic, prescribed** demonstration value; K, lambda, H are nominal. The DEM, hillshade and slope are **plotting-only context**: no solver input reads them.

**Bounded run.** 500 m grid, standard (8 km) outer domain, four winds, a 2 km small outer domain, and a mesh comparison (500 m dt 100 s, 500 m dt 50 s, 250 m dt 50 s). The 250 m comparison uses a **short window** (T = 2 h, 3 km buffer crop, the smoke profile) to stay within minutes; the full 8 h comparison is produced by the command-line run `python -m pravah benchmark --config configs/belauri_forward.json`.

Terms: *local* = emitted inside the municipality (road + built-up layers), *external* = the external patch (outside it). Origin follows the source layer, not where the mass is now. Local layers are clipped to the polygon, boundary cells count in full, so ~5 % of a local layer's mass lies in cells that straddle the border (documented tolerance).

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from pathlib import Path
from pravah.config import load_config
from pravah import belauri as B
from pravah.geo.scenarios import numerical_diffusion_estimate, LABEL
%matplotlib inline
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
T0 = time.perf_counter()
root = Path.cwd()
while not (root / "configs" / "belauri_forward.json").exists() and root != root.parent:
    root = root.parent
cfg = load_config(root / "configs" / "belauri_forward.json")
cfg["manifest_note"] = "path resolved by pravah"
cfg["wind"]["run"] = ["from_west_southwest", "from_north", "reversal", "channelled_demo"]
study = B.Study(cfg)
print(LABEL)

## 1. Provenance panel (from the manifest)

In [ ]:
m = study.full.manifest
ds = m["datasets"]
prov = pd.DataFrame([{"id": d["id"], "role": d.get("role"), "licence": str(d["license"])[:70], "retrieved_utc": d["retrieved_utc"]} for d in ds])
display(prov)
print("boundary_status:", m["boundary_status"], "| CRS EPSG", m["crs"]["epsg"], "| manifest sha256", study.manifest_sha256[:16], "...")
print("processed-array licences:", sorted({str(p["license"])[:60] for p in m["processed"]})[:3])
print("resolutions (m):", m["grid_config"]["resolutions_m"], "| buffer (m):", m["grid_config"]["buffer_m"])
print("warnings:"); [print("  -", w) for w in m["warnings"]]
print("assumptions:"); [print("  -", a) for a in m["assumptions"]]
ep = m["external_patch"]
print(f"external patch: centre (local m) {ep['centre_local_m']}, sigma {ep['sigma_m']} m, {ep['distance_to_polygon_km']:.2f} km from the polygon; 'international' is NOT asserted (patch is in no ADM0 polygon).")

## 2. The region: boundary, surrounding domain, hillshade, N-S profile, sources, report points

In [ ]:
fig = B.fig_overview(study); plt.show()
lv = study.region.level(500.0)
print("domain", lv.grid.nx, "x", lv.grid.ny, "cells of", lv.grid.dx, "m =", lv.grid.lx / 1000, "x", lv.grid.ly / 1000, "km; polygon cells (500 m):", lv.info["polygon_cells"])
print(pd.DataFrame(study.points).round(0))

The hillshade and the elevation profile are drawn for orientation only; relief in the municipality is about 37 m, so even a physical use would be weak. **No number below depends on them** (checked in section 9).

## 3. Choices: domain and time
Travel distance u T = 1.5 m/s x 28800 s = 43 km (larger than the 33 km domain: plumes can leave it, which is fine with open boundaries); physical diffusion length sqrt(2 K T) = 2.4 km; removal lifetime 1/lambda = 6 h; Courant number u dt / dx = 0.3 at 500 m / 100 s and 0.3 at 250 m / 50 s.

In [ ]:
u, K, T = study.speed, study.K, study.T
print(f"u*T = {u*T/1000:.0f} km | sqrt(2KT) = {np.sqrt(2*K*T)/1000:.1f} km | 1/lambda = {1/study.lam/3600:.0f} h | Courant 500 m: {u*100/500:.2f}, 250 m: {u*50/250:.2f}")
rows = [{"mesh": f"{dx:g} m, dt {dt:g} s", "Pe=u dx/K": u*dx/K, "excess K_eff/K-1": numerical_diffusion_estimate(u, dx, dt, K)} for dx, dt in ((500, 100), (500, 50), (250, 50))]
display(pd.DataFrame(rows).round(2))
print("The discretisation adds several times the nominal K = 100 m^2/s at these cells; K is a NOMINAL input, the effective spreading is larger (estimate exact for axis-aligned wind).")

## 4. Local-only, external-only and combined fields for several winds

In [ ]:
t0 = time.perf_counter()
res = {n: study.wind_result(n) for n in cfg["wind"]["run"]}
print(f"solved in {time.perf_counter()-t0:.0f} s")
rows = []
for n, r in res.items():
    s = r.summary
    rows.append({"wind": n, "local peak": s["local"]["peak"], "external peak": s["external"]["peak"], "combined peak": s["total"]["peak"],
                 "combined regional mean": s["total"]["time_mean_regional_mean"], "external fraction": s["external_fraction"],
                 "max |direct - (local+external)|": r.consistency["max_abs_err_direct_vs_tag_sum"], "budget residual": r.consistency["max_rel_budget_residual"]})
display(pd.DataFrame(rows).round(4))
print("emission amplitudes (SYNTHETIC, ug/s):", study.amplitudes)

The first column pair checks the linear tag machinery: the combined direct solve equals local + external to round-off, and the mass budget closes. All winds are prescribed and synthetic; `channelled_demo` is a divergence-free stream function (uniform wind plus a Gaussian eddy), **not derived from the terrain**.

## 5. Snapshots (common colour scale) and an animation

In [ ]:
ref = res["from_west_southwest"]
fig = B.fig_contributions(study, ref, [10800.0, 18000.0, 28800.0]); plt.show()

In [ ]:
try:
    from IPython.display import HTML
    anim = B.animate_on_map(study, ref, "total", max_frames=49)
    html = anim.to_jshtml(); plt.close("all")
    display(HTML(html))
except Exception as exc:
    print("animation skipped:", type(exc).__name__, exc)

## 6. Arrival of the external tag, source contributions and regional summaries

In [ ]:
fig = B.fig_arrival(study, list(res.values())); plt.show()
arr = pd.DataFrame([{"wind": n, **a} for n, r in res.items() for a in r.arrival])
display(arr[["wind", "point", "arrival_s", "arrival_after_onset_s", "max_external"]].round(4))
print(f"arrival = first time the external-tag concentration exceeds {study.arrival_level} ug/m3 (output interval {study.out_dt:.0f} s, linear interpolation); the patch switches on at {study.onset_s:.0f} s; blank = never reached in this run.")

In [ ]:
fig = B.fig_regional(list(res.values())); plt.show()
lay = res["from_west_southwest"]
comp = pd.DataFrame({k: [lay.resp.g[j][lay.resp.times >= study.t_exclude].max() * lay.case.theta[j]] for j, k in enumerate(lay.resp.param_names)}, index=["peak anywhere, one source at a time (ug/m3)"])
display(comp.round(4))

## 7. Smaller and larger outer domain

In [ ]:
t0 = time.perf_counter()
buf = float(cfg["region"]["small_buffer_m"])
orows, oarr = [], []
for n in ("from_west_southwest", "reversal"):
    row, arr_, f, case = B.outer_domain_rows(study, res[n], buf)
    orows.append(row); oarr += arr_
display(pd.DataFrame(orows)[["wind", "buffer_m", "n_cells_small", "n_cells_standard", "peak_change_pct", "mean_change_pct", "rmse_total_rel_peak", "max_abs_diff_whole_small_domain"]].round(5))
display(pd.DataFrame(oarr).round(1))
print(f"{time.perf_counter()-t0:.0f} s. Differences are the effect of the open boundary being closer. They vanish for the steady WSW wind (plumes leave through the boundary) but are not small for the reversal wind (about -10 % in the regional mean): mass that left a smaller domain cannot return when the wind turns.")

## 8. Two mesh resolutions and the time step

In [ ]:
t0 = time.perf_counter()
scfg = load_config(root / "configs" / "belauri_forward_smoke.json")   # short window: T = 2 h, 3 km buffer crop
sstudy = B.Study(scfg)
base = sstudy.wind_result("from_west_southwest")
mrows, marr = B.mesh_rows(sstudy, "from_west_southwest", scfg["mesh_sensitivity"]["runs"], scfg["mesh_sensitivity"]["reference"], base)
display(pd.DataFrame(mrows)[["run", "courant_u_dt_over_dx", "mean_change_vs_ref_pct", "peak_change_vs_ref_pct", "rmse_total_rel_ref_peak", "solve_wall_s", "numerical_excess_Keff_over_K_minus_1", "K_effective_m2_s_estimate"]].round(3))
display(pd.DataFrame(marr).round(1))
print(f"{time.perf_counter()-t0:.0f} s. 250 m is compared on the 500 m grid by block averaging. Numerical diffusion (K_eff) is reported separately from the physical K = {sstudy.K:g}.")

## 9. What changes physically, and what is only plotting
**Physical inputs** (change the numbers): wind, K, removal rate lambda, emission amplitudes, the source geometry (which layer carries the emission), the outer-domain size (through the open boundary), the mesh and the time step. **Plotting-only layers**: DEM, valid fraction, hillshade, slope and the elevation profiles. The check below replaces them with constants and re-solves.

In [ ]:
t0 = time.perf_counter()
chk = B.plotting_only_check(study.region, lambda r: study.case("from_west_southwest", region=r))
a, _ = study.direct(chk["case_a"], "orig"); b, _ = study.direct(chk["case_b"], "scrambled")
print("layers replaced:", chk["layers"])
print("scenario identical:", chk["scenario_identical"], "| max |change in concentration| =", float(np.abs(a - b).max()))
sens = {"K_factors": [0.5, 2.0], "lam_factors": [2.0], "amplitude_factor": 2.0}
wc = pd.DataFrame(B.what_changes(study, ref, res, [], [], None, sens))
display(wc[["input", "variation", "mean_change_pct", "peak_change_pct", "external_fraction_change", "solves"]].round(3))
print(f"{time.perf_counter()-t0:.0f} s")

Reading the table: doubling an amplitude scales that tag exactly (linearity); the wind and the source geometry change the pattern most; K changes the totals little because (as section 3 shows) the scheme's own diffusion at 500 m cells dominates the nominal K; the DEM-type layers change **nothing**.

In [ ]:
print(f'total notebook time {time.perf_counter()-T0:.0f} s')